In [19]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

os.makedirs("../outputs/tables", exist_ok=True)

In [20]:
# Veriyi yükle
df = pd.read_csv("../data/train.csv")

print("Ham veri boyutu:", df.shape)
df.head()

Ham veri boyutu: (103904, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,2,2,2,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,4,5,5,3,3,4,4,3,3,3,0,0.0,satisfied


In [21]:
# Gereksiz sütunları kaldır
df = df.drop(columns=["Unnamed: 0", "id"])

print("Sütun silme sonrası veri boyutu:", df.shape)
print("\nKalan sütunlar:")
print(df.columns.tolist())

Sütun silme sonrası veri boyutu: (103904, 23)

Kalan sütunlar:
['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'satisfaction']


In [22]:
# Eksik veriyi doldur
arrival_delay_median = df["Arrival Delay in Minutes"].median() # eksik değerleri medyan ile dolduruyor

df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(arrival_delay_median)

print("Eksik veri sayıları:")
print(df.isnull().sum())

Eksik veri sayıları:
Gender                               0
Customer Type                        0
Age                                  0
Type of Travel                       0
Class                                0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
Ease of Online booking               0
Gate location                        0
Food and drink                       0
Online boarding                      0
Seat comfort                         0
Inflight entertainment               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Inflight service                     0
Cleanliness                          0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
satisfaction                         0
dtype: int64


In [23]:
# eksik değerleri medyan ile dolduruyor
df["satisfaction"] = df["satisfaction"].map({
    "neutral or dissatisfied": 0,
    "satisfied": 1
})

print("Hedef değişken dağılımı:")
print(df["satisfaction"].value_counts())

print("\nHedef değişken benzersiz değerleri:")
print(df["satisfaction"].unique())

Hedef değişken dağılımı:
satisfaction
0    58879
1    45025
Name: count, dtype: int64

Hedef değişken benzersiz değerleri:
[0 1]


In [24]:
# Temizlenmiş veriyi kontrol et
print("Temizlenmiş veri boyutu:", df.shape)
df.head()

Temizlenmiş veri boyutu: (103904, 23)


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,0
1,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,0
2,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,1
3,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,2,2,2,2,2,5,3,1,4,2,11,9.0,0
4,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,4,5,5,3,3,4,4,3,3,3,0,0.0,1


In [25]:
# giriş özellikleri X (modele verdiğimiz bilgiler)
# hedef değişken y (modelin tahmin etmeye çalıştığı sonuç)

# “Bana yolcuya ait bilgileri ver, ben bu yolcunun memnuniyet durumunu tahmin edeyim.”

X = df.drop("satisfaction", axis=1)
y = df["satisfaction"]

print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

X boyutu: (103904, 22)
y boyutu: (103904,)


In [26]:
# Kategorik ve sayısal sütunları ayır
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "string"]).columns.tolist()

print("Kategorik sütunlar:")
print(categorical_features)

print("\nSayısal sütunlar:")
print(numeric_features)

Kategorik sütunlar:
['Gender', 'Customer Type', 'Type of Travel', 'Class']

Sayısal sütunlar:
['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


In [27]:
# temizlenmiş veriyi kaydetmek için
df.to_csv("../outputs/tables/train_clean.csv", index=False)
print("Temizlenmiş veri önizlemesi kaydedildi.")

Temizlenmiş veri önizlemesi kaydedildi.
